# 06 - HistGradientBoosting tuning, U.S. training only

This notebook tunes HistGradientBoosting on the official U.S. training rows. It uses the conservative feature policy and five-fold development CV. The reserved U.S. IDs and labels are not loaded. Search scores are tuning estimates because the best settings are selected from these same folds.


In [1]:
from pathlib import Path
import sys, time, warnings
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'preprocessing.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
from utils.us_modeling import (CONSERVATIVE_POLICY, development_cv, load_us_training_data,
    run_randomized_search, scoring, search_result, conservative_pipeline)
SEED = 42
X_train, y_train, project_ids = load_us_training_data(ROOT)
assert len(project_ids) == 26945 and project_ids.is_unique
assert np.isfinite(y_train).all() and set(y_train.unique()) == {0, 1}
assert CONSERVATIVE_POLICY == dict(include_timing=False, include_flags=False, include_reward=False, exclude_renamed=True)
baseline = conservative_pipeline(HistGradientBoostingClassifier(max_iter=200, learning_rate=0.06, random_state=SEED), 'target')
base_scores = cross_validate(baseline, X_train, y_train, cv=development_cv(), scoring=scoring(), n_jobs=1, return_train_score=True, error_score='raise')
baseline_row = {m: float(base_scores[f'test_{m}'].mean()) for m in scoring()}
baseline_row['roc_auc_sd'] = float(base_scores['test_roc_auc'].std())
print(f'official U.S. training rows: {len(X_train):,}; success rate: {y_train.mean():.4f}')
print('baseline:', {k: round(v, 4) for k, v in baseline_row.items()})


official U.S. training rows: 26,945; success rate: 0.5461
baseline: {'roc_auc': 0.7235, 'accuracy': 0.6684, 'f1': 0.7156, 'roc_auc_sd': 0.0082}


In [2]:
model = HistGradientBoostingClassifier(random_state=SEED)
space = {
    'model__learning_rate': loguniform(0.015, 0.15),
    'model__max_iter': randint(120, 401),
    'model__max_leaf_nodes': randint(7, 33),
    'model__max_depth': [None, 3, 5, 7],
    'model__min_samples_leaf': randint(20, 121),
    'model__l2_regularization': loguniform(1e-3, 20),
}
search, trials = run_randomized_search(X_train, y_train, model, 'target', space, n_iter=15)
result = search_result(search, trials, 'HistGradientBoosting')
result['baseline_roc_auc'] = baseline_row['roc_auc']
result_table = pd.DataFrame([result]).drop(columns=['best_params'])
display(result_table.round(4))
print('best parameters:', result['best_params'])
display(trials.head(10).drop(columns=['params']).round(4))


,model,cohort,feature_policy,cv,search_iterations,estimate,cv_roc_auc,cv_roc_auc_sd,train_roc_auc,overfit_gap,accuracy,f1,fit_seconds,elapsed_seconds,baseline_roc_auc
0,HistGradientBoosting,official U.S. training rows,"conservative: no timing, flags, rewards, or re...",5-fold StratifiedKFold shuffle=True random_sta...,15,ordinary tuning CV; optimistic selection estimate,0.7255,0.0087,0.7464,0.0209,0.6694,0.7166,0.4427,71.4373,0.7235


best parameters: {'l2_regularization': np.float64(0.0026915045092098205), 'learning_rate': np.float64(0.04318571864689941), 'max_depth': None, 'max_iter': 219, 'max_leaf_nodes': 14, 'min_samples_leaf': 43}


,trial_rank,cv_roc_auc,cv_roc_auc_sd,train_roc_auc,cv_accuracy,cv_f1,mean_fit_seconds,overfit_gap,elapsed_seconds
0,1,0.7255,0.0087,0.7464,0.6694,0.7166,0.4427,0.0209,71.4373
1,2,0.7254,0.0083,0.7465,0.6720,0.7188,0.6468,0.0212,71.4373
2,3,0.7254,0.0085,0.7459,0.6716,0.7182,0.2498,0.0206,71.4373
3,4,0.7251,0.0085,0.7457,0.6702,0.7169,0.3739,0.0207,71.4373
4,5,0.7249,0.0076,0.7375,0.6712,0.7179,0.5376,0.0127,71.4373
5,6,0.7249,0.0081,0.7479,0.6695,0.7166,0.6119,0.0230,71.4373
6,7,0.7248,0.0085,0.7495,0.6701,0.7166,0.3993,0.0247,71.4373
7,8,0.7246,0.0086,0.7515,0.6693,0.7157,0.5318,0.0269,71.4373
8,9,0.7244,0.0082,0.7475,0.6702,0.7174,0.8075,0.0230,71.4373
9,10,0.7244,0.0082,0.7504,0.6721,0.7179,0.2848,0.0260,71.4373


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(trials['trial_rank'], trials['cv_roc_auc'], 'o-', label='validation ROC-AUC')
axes[0].plot(trials['trial_rank'], trials['train_roc_auc'], 'o-', label='training ROC-AUC')
axes[0].set_xlabel('Search rank'); axes[0].set_ylabel('ROC-AUC'); axes[0].set_title('HGB tuning trials'); axes[0].legend()
axes[1].scatter(trials['mean_fit_seconds'], trials['cv_roc_auc'], c=trials['overfit_gap'], cmap='viridis')
axes[1].set_xlabel('Mean fit seconds'); axes[1].set_ylabel('Validation ROC-AUC'); axes[1].set_title('Cost and train-validation gap')
plt.tight_layout()
print(f"baseline AUC {baseline_row['roc_auc']:.4f}; tuned AUC {result['cv_roc_auc']:.4f}; difference {result['cv_roc_auc'] - baseline_row['roc_auc']:+.4f}")
print(f"best-trial train-validation gap: {result['overfit_gap']:.4f}; this is a diagnostic, not an independent estimate")


baseline AUC 0.7235; tuned AUC 0.7255; difference +0.0020
best-trial train-validation gap: 0.0209; this is a diagnostic, not an independent estimate


The baseline and tuning results above are training-only five-fold results. A small change should not be treated as a final model-selection decision. Final holdout evaluation remains gated by the shared evaluation policy.
